## accounts_userquestionrecord 투표 기록 테이블 전처리 확인

In [1]:
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"
TABLE_NAME = "accounts_userquestionrecord"

client = bigquery.Client(project=PROJECT_ID)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.{TABLE_NAME}`
"""

df = client.query(sql).to_dataframe()
df.head()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,status,created_at,chosen_user_id,question_id,user_id,question_piece_id,has_read,answer_status,answer_updated_at,report_count,opened_times
0,945319,I,2023-04-29 13:22:05+00:00,849995,132,851717,1213085,1,P,2023-05-06 10:31:30+00:00,0,3
1,978922,I,2023-04-29 14:49:06+00:00,849922,180,849450,1235436,1,N,2023-04-29 14:49:06+00:00,0,3
2,1095692,I,2023-04-30 03:29:48+00:00,850031,132,850229,1395859,1,N,2023-04-30 03:29:48+00:00,0,3
3,1167181,I,2023-04-30 07:43:10+00:00,856172,116,857422,1509130,1,N,2023-04-30 07:43:10+00:00,0,3
4,1171173,I,2023-04-30 07:58:36+00:00,855039,132,855117,1511169,1,N,2023-04-30 07:58:36+00:00,0,3


## 결측치 및 데이터 정보 확인

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1217558 entries, 0 to 1217557
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype              
---  ------             --------------    -----              
 0   id                 1217558 non-null  Int64              
 1   status             1217558 non-null  str                
 2   created_at         1217558 non-null  datetime64[us, UTC]
 3   chosen_user_id     1217558 non-null  Int64              
 4   question_id        1217558 non-null  Int64              
 5   user_id            1217558 non-null  Int64              
 6   question_piece_id  1217558 non-null  Int64              
 7   has_read           1217558 non-null  Int64              
 8   answer_status      1217558 non-null  str                
 9   answer_updated_at  1217558 non-null  datetime64[us, UTC]
 10  report_count       1217558 non-null  Int64              
 11  opened_times       1217558 non-null  Int64              
dtypes: Int64(8), datetime64[u

In [4]:
df.isna().sum()

id                   0
status               0
created_at           0
chosen_user_id       0
question_id          0
user_id              0
question_piece_id    0
has_read             0
answer_status        0
answer_updated_at    0
report_count         0
opened_times         0
dtype: int64

## 중복값 확인

In [5]:
print("전체 행 중복:", df.duplicated().sum())
print("id 중복:", df["id"].duplicated().sum())

전체 행 중복: 0
id 중복: 0


## 날짜 범위 및 미래 날짜 확인

In [6]:
date_cols = ["created_at", "answer_updated_at"]
display(df[date_cols].agg(["min", "max"]))

now_utc = pd.Timestamp.now(tz="UTC")
for col in date_cols:
    print(f"{col} 미래 날짜: {(df[col] > now_utc).sum()}건")

,created_at,answer_updated_at
min,2023-04-28 12:27:49+00:00,2023-04-28 12:27:49+00:00
max,2024-05-08 01:36:18+00:00,2024-05-08 01:36:18+00:00


created_at 미래 날짜: 0건
answer_updated_at 미래 날짜: 0건


## 수치형 컬럼 이상치 확인

In [7]:
df[["opened_times", "report_count"]].describe()

,opened_times,report_count
count,1217558.0,1217558.0
mean,0.063325,0.000177
std,0.301147,0.020405
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,3.0,14.0


In [8]:
# 횟수형 컬럼의 음수 여부
(df[["opened_times", "report_count"]] < 0).sum()

opened_times    0
report_count    0
dtype: Int64

## 범주형 컬럼 확인

In [9]:
for col in ["status", "answer_status", "has_read"]:
    print(f"[{col}]")
    print(df[col].value_counts(dropna=False), "\n")

[status]
status
C    1156322
I      60578
B        658
Name: count, dtype: int64 

[answer_status]
answer_status
N    1097932
A     111761
P       7865
Name: count, dtype: int64 

[has_read]
has_read
1    675931
0    541627
Name: count, dtype: Int64 



- `status`: C(닫힘), I(초성 열림), B(차단)
- `answer_status`: N(미답변), P(비공개), A(공개)
- `has_read`: 투표 열람 여부

## 시간 순서 확인

In [10]:
# 답변 시간이 투표 생성 시간보다 이른 행과 시간 차이
invalid_time = (
    df["answer_updated_at"].notna()
    & (df["answer_updated_at"] < df["created_at"])
)
time_gap = (
    df.loc[invalid_time, "created_at"]
    - df.loc[invalid_time, "answer_updated_at"]
).dt.total_seconds()

print("시간 역전 행:", invalid_time.sum())
print(time_gap.describe())

시간 역전 행: 1426
count    1426.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64


## 전처리 확인 결과

- `id`를 기준으로 중복 여부를 확인한다.
- 날짜 최솟값·최댓값과 미래 날짜 여부를 확인한다.
- 횟수형 컬럼의 음수, 상태 코드, 시간 역전 여부와 차이를 확인한다.
- 데이터 수정 여부는 확인 결과를 바탕으로 팀 협의 후 결정한다.